In [1]:
import os
import pandas as pd
os.chdir("/Users/debapriyaroy/Documents/GitHub/Manufacturing-Performance-Analytics")

prod = pd.read_csv("data/raw/mes_production_hourly.csv")
downtime = pd.read_csv("data/raw/mes_downtime_events.csv")
rejects = pd.read_csv("data/raw/qms_rejects_by_shift.csv")
erp = pd.read_excel("data/raw/erp_material_master.xlsx", skiprows=3)

print("Rows before cleaning:")
print("production:", len(prod))
print("downtime:", len(downtime))
print("rejects:", len(rejects))
print("erp:", len(erp))

Rows before cleaning:
production: 14300
downtime: 13741
rejects: 1780
erp: 17


**Fixing Line Names**

In [2]:
def clean_line(col):
    return "L" + col.str.strip().str[-1]

prod["Line"] = clean_line(prod["Line"])
downtime["Line"] = clean_line(downtime["Line"])
rejects["Line"] = clean_line(rejects["Line"])

print(prod["Line"].value_counts())

Line
L2    3581
L4    3576
L3    3572
L1    3571
Name: count, dtype: int64


**Fixing Shift Codes**

In [3]:
prod["Shift_Code"] = prod["Shift_Code"].str.strip().str.upper()

print(prod["Shift_Code"].value_counts())

Shift_Code
A    5050
B    5046
C    4204
Name: count, dtype: int64


**Removing Duplicates**

In [4]:
print("Duplicates found:", prod.duplicated().sum())

prod = prod.drop_duplicates()

print("Duplicates left:", prod.duplicated().sum())
print("Production rows now:", len(prod))

Duplicates found: 60
Duplicates left: 0
Production rows now: 14240


**Hour_Start made a real date**

In [5]:
prod["Hour_Start"] = pd.to_datetime(prod["Hour_Start"], format="%d/%m/%Y %H:%M")

print(prod["Hour_Start"].dtype)
print("First:", prod["Hour_Start"].min())
print("Last:", prod["Hour_Start"].max())

datetime64[us]
First: 2026-04-01 06:00:00
Last: 2026-10-01 05:00:00


**Adding Production Date column**


In [6]:
prod["ProductionDate"] = (prod["Hour_Start"] - pd.Timedelta(hours=6)).dt.date

prod = prod.sort_values(["Line", "Hour_Start"])
print(prod[["Hour_Start", "Shift_Code", "ProductionDate"]].tail(30))

               Hour_Start Shift_Code ProductionDate
3481  2026-09-30 00:00:00          C     2026-09-29
6115  2026-09-30 01:00:00          C     2026-09-29
1524  2026-09-30 02:00:00          C     2026-09-29
7299  2026-09-30 03:00:00          C     2026-09-29
335   2026-09-30 04:00:00          C     2026-09-29
11061 2026-09-30 05:00:00          C     2026-09-29
13761 2026-09-30 06:00:00          A     2026-09-30
11303 2026-09-30 07:00:00          A     2026-09-30
12138 2026-09-30 08:00:00          A     2026-09-30
2674  2026-09-30 09:00:00          A     2026-09-30
70    2026-09-30 10:00:00          A     2026-09-30
6250  2026-09-30 11:00:00          A     2026-09-30
3586  2026-09-30 12:00:00          A     2026-09-30
4850  2026-09-30 13:00:00          A     2026-09-30
12949 2026-09-30 14:00:00          B     2026-09-30
3568  2026-09-30 15:00:00          B     2026-09-30
8387  2026-09-30 16:00:00          B     2026-09-30
6928  2026-09-30 17:00:00          B     2026-09-30
11463 2026-0

**Coverting downtime times to real dates**

In [7]:
downtime["Start_Time"] = pd.to_datetime(downtime["Start_Time"])
downtime["End_Time"] = pd.to_datetime(downtime["End_Time"])

print(downtime[["Start_Time", "End_Time"]].dtypes)
print("Blank End_Time:", downtime["End_Time"].isna().sum())

Start_Time    datetime64[us]
End_Time      datetime64[us]
dtype: object
Blank End_Time: 82


**Filling the 82 blank End_Times**

In [8]:
blank = downtime["End_Time"].isna()
print("Blank End_Time before:", blank.sum())

downtime.loc[blank, "End_Time"] = downtime.loc[blank, "Start_Time"] + pd.to_timedelta(downtime.loc[blank, "Duration_Min_Reported"], unit="min")

print("Blank End_Time after:", downtime["End_Time"].isna().sum())

Blank End_Time before: 82
Blank End_Time after: 0


In [9]:
bad = downtime["End_Time"] < downtime["Start_Time"]
print("Stops ending before they start:", bad.sum())
print(downtime.loc[bad, "Operator_Comment"].unique())

downtime = downtime[~bad]
print("Downtime rows now:", len(downtime))

Stops ending before they start: 25
<StringArray>
['clock sync error']
Length: 1, dtype: str
Downtime rows now: 13716


**Removing stops that were logged twice**

In [10]:
key = ["Line", "Machine_ID", "Start_Time", "Reason_Code"]
print("Double-logged stops:", downtime.duplicated(subset=key).sum())

downtime = downtime.sort_values("Event_ID").drop_duplicates(subset=key)

print("Double-logged left:", downtime.duplicated(subset=key).sum())
print("Downtime rows now:", len(downtime))

Double-logged stops: 90
Double-logged left: 0
Downtime rows now: 13626


**Calculating how long each stop lasted**


In [11]:
downtime["DurationMinutes"] = (downtime["End_Time"] - downtime["Start_Time"]).dt.total_seconds() / 60

print("Blank durations:", downtime["DurationMinutes"].isna().sum())
print("Shortest stop:", downtime["DurationMinutes"].min(), "min")
print("Longest stop:", downtime["DurationMinutes"].max(), "min")
print("Total downtime:", downtime["DurationMinutes"].sum(), "min")

Blank durations: 0
Shortest stop: 2.0 min
Longest stop: 198.0 min
Total downtime: 204574.0 min


**Give each stop a production day and a shift**

In [18]:
downtime["ProductionDate"] = (downtime["Start_Time"] - pd.Timedelta(hours=6)).dt.date

def shift_of(t):
    if 6 <= t.hour < 14:
        return "A"
    elif 14 <= t.hour < 22:
        return "B"
    else:
        return "C"

downtime["Shift"] = downtime["Start_Time"].apply(shift_of)

print(downtime["Shift"].value_counts())
print(downtime[["Start_Time", "ProductionDate", "Shift"]].head(100))

Shift
B    4635
C    4519
A    4472
Name: count, dtype: int64
             Start_Time ProductionDate Shift
0   2026-04-01 06:00:00     2026-04-01     A
3   2026-04-01 06:00:00     2026-04-01     A
1   2026-04-01 06:00:00     2026-04-01     A
4   2026-04-01 07:08:00     2026-04-01     A
5   2026-04-01 07:17:00     2026-04-01     A
..                  ...            ...   ...
96  2026-04-02 03:47:00     2026-04-01     C
97  2026-04-02 03:58:00     2026-04-01     C
98  2026-04-02 04:05:00     2026-04-01     C
99  2026-04-02 04:19:00     2026-04-01     C
100 2026-04-02 04:43:00     2026-04-01     C

[100 rows x 3 columns]


In [19]:
rejects.head(2)

,Prod_Date,Shift,Line,Material_No,Seal_Integrity,Weight_Out_Of_Spec,Label_Code_Error,Metal_Detect,Damaged_Pack
0,2026-04-01,A,L1,SB-101,125,459,65,14,81
1,2026-04-01,B,L1,SB-104,123,443,68,8,106


**Rearranging the rejects table so that the reject reason becomes something we can filter and count**

In [23]:
id_cols = ["Prod_Date", "Shift", "Line", "Material_No"]
rejects_long = rejects.melt(id_vars=id_cols, var_name="Reason", value_name="Packs")

print("Rows before:", len(rejects), "| after melt:", len(rejects_long))
print("Total packs before:", rejects.drop(columns=id_cols).sum().sum())
print("Total packs after:", rejects_long["Packs"].sum())

Rows before: 1780 | after melt: 8900
Total packs before: 727540
Total packs after: 727540


In [26]:
rejects_long.head(10)

,Prod_Date,Shift,Line,Material_No,Reason,Packs
0,2026-04-01,A,L1,SB-101,Seal_Integrity,125
1,2026-04-01,B,L1,SB-104,Seal_Integrity,123
2,2026-04-01,C,L1,SB-104,Seal_Integrity,91
3,2026-04-01,A,L2,PC-204,Seal_Integrity,75
4,2026-04-01,B,L2,PC-201,Seal_Integrity,95
5,2026-04-01,C,L2,PC-204,Seal_Integrity,93
6,2026-04-01,A,L3,RM-302,Seal_Integrity,82
7,2026-04-01,B,L3,RM-303,Seal_Integrity,103
8,2026-04-01,C,L3,RM-303,Seal_Integrity,103
9,2026-04-01,A,L4,SJ-402,Seal_Integrity,47


**Adding profit per pack to the ERP table**

In [27]:
erp["Profit_per_pack_USD"] = erp["Net_Price_USD"] - erp["Std_Cost_USD"]

print(erp[["Material_No", "Std_Cost_USD", "Net_Price_USD", "Profit_per_pack_USD"]].head(5))
print("Products:", len(erp), "| any negative profit?", (erp["Profit_per_pack_USD"] < 0).any())

  Material_No  Std_Cost_USD  Net_Price_USD  Profit_per_pack_USD
0      SB-101          0.21           0.58                 0.37
1      SB-102          0.33           0.79                 0.46
2      SB-103          0.29           0.71                 0.42
3      SB-104          0.14           0.45                 0.31
4      PC-201          1.05           2.05                 1.00
Products: 17 | any negative profit? False


**Save the clean tables as new files**

In [29]:
os.makedirs("data/processed", exist_ok=True)

prod.to_csv("data/processed/production_hourly_clean.csv", index=False)
downtime.to_csv("data/processed/downtime_events_clean.csv", index=False)
rejects_long.to_csv("data/processed/rejects_clean.csv", index=False)
erp.to_csv("data/processed/material_master_clean.csv", index=False)

print("Saved. Rows: production", len(prod), "| downtime", len(downtime), "| rejects", len(rejects_long), "| products", len(erp))

Saved. Rows: production 14240 | downtime 13626 | rejects 8900 | products 17
